# 영상 채널 성과 분석 (2019–2021)

직접 운영한 영상 채널의 YouTube Studio 데이터를 **노출 → 클릭 → 시청 → 구독** 퍼널로 분석한 노트북입니다.

- 분석 기간: 2019-01-13 ~ 2021-12-31 (채널 누적 조회수의 95.5%가 발생한 기간)
- 원본 CSV(`data/raw/`)는 영상 제목·ID가 포함되어 있어 공개 저장소에 올리지 않습니다. 이 노트북은 원본을 읽은 직후 식별 정보를 제거하고, 익명화된 결과만 `data/processed/`에 저장합니다.
- 표기 원칙: **[사실]** 데이터로 확인된 것 / **[추정]** 근거가 있으나 직접 확인 불가 / **[가설]** 검증되지 않은 해석

In [1]:
import pandas as pd
RAW, OUT = 'data/raw', 'data/processed'
START, END = '2019-01-13', '2021-12-31'
SPIKE = ('2019-03-09', '2019-03-22')   # 급증 구간
pd.set_option('display.width', 140)

## 1. 데이터 준비와 익명화

영상은 제목·ID 대신 **영상 길이(초)**로 매칭해 유형 코드를 붙입니다. 공개 상태 이력은 일별 조회수가 장기간 0으로 떨어지는 시점과 운영자 기록으로 확인했습니다.

In [2]:
VIDEO_MAP = {  # 길이(초): (코드, 유형, 공개 상태 이력)
    2131: ('L1', '롱폼 음성 콘텐츠', '상시 공개'),
    1353: ('L2', '롱폼 음성 콘텐츠', '2025-07 비공개 → 2026 재공개'),
    2260: ('L3', '롱폼 음성 콘텐츠', '2025-06 비공개 → 2026 재공개'),
    231:  ('S1', '콘셉트 숏폼(영상통화 형식)', '상시 공개'),
    64:   ('C1', '예능 클립', '2023-10 비공개 → 2026 재공개'),
    51:   ('C2', '예능 클립', '비공개 이력(시점 미확인)'),
    374:  ('M1', '편집 영상', '비공개 이력(시점 미확인)'),
    141:  ('M2', '편집 영상', '비공개 이력(시점 미확인)'),
    3111: ('P1', '플레이리스트', '저작권 부분 차단 → 비공개'),
}
def hms(s):
    h, m, sec = map(int, str(s).split(':')); return h*3600 + m*60 + sec

v = pd.read_csv(f'{RAW}/videos_2019_2021.csv')
total = v[v['콘텐츠'] == '합계'].iloc[0]
v = v[v['콘텐츠'] != '합계'].copy()
v[['code', 'type', 'status']] = v['길이'].map(VIDEO_MAP).apply(pd.Series)
v = v.rename(columns={'길이': 'length_s', '유효 조회수': 'views', '시청 시간(단위: 시간)': 'watch_h',
                      '구독자': 'subs', '썸네일 노출수': 'impr', '썸네일 클릭률 (%)': 'ctr',
                      '평균 조회율 (%)': 'pct_viewed'})
v['avg_view_s'] = v['평균 시청 지속 시간'].map(hms)
v['published'] = pd.to_datetime(v['동영상 게시 시간'], format='%b %d, %Y').dt.date
v = v[['code', 'type', 'status', 'published', 'length_s', 'impr', 'ctr', 'views', 'avg_view_s', 'pct_viewed', 'watch_h', 'subs']]
v.sort_values('views', ascending=False).reset_index(drop=True)

,code,type,status,published,length_s,impr,ctr,views,avg_view_s,pct_viewed,watch_h,subs
0,L1,롱폼 음성 콘텐츠,상시 공개,2019-01-19,2131.0,127652,17.04,25600,226,10.63,1610.7941,31
1,S1,콘셉트 숏폼(영상통화 형식),상시 공개,2019-07-13,231.0,65837,19.12,14863,59,25.71,245.2125,29
2,L2,롱폼 음성 콘텐츠,2025-07 비공개 → 2026 재공개,2019-07-30,1353.0,44764,8.72,4836,210,15.53,282.2400,13
3,L3,롱폼 음성 콘텐츠,2025-06 비공개 → 2026 재공개,2019-07-28,2260.0,29628,11.90,4114,273,12.10,312.4636,3
4,C1,예능 클립,2023-10 비공개 → 2026 재공개,2019-08-11,64.0,12869,6.39,1005,50,79.11,14.1352,1
5,M2,편집 영상,비공개 이력(시점 미확인),2020-09-05,141.0,1972,11.46,266,40,28.50,2.9688,0
6,C2,예능 클립,비공개 이력(시점 미확인),2019-08-25,51.0,5352,3.59,249,32,63.21,2.2296,1
7,M1,편집 영상,비공개 이력(시점 미확인),2019-07-22,374.0,4092,3.32,205,62,16.72,3.5614,0
8,P1,플레이리스트,저작권 부분 차단 → 비공개,2021-07-23,3111.0,238,5.04,30,52,1.70,0.4400,-1


### 비어 있는 지표 제외
순 시청자수·신규/재방문 시청자·고정 시청자·공동 시청은 이 기간 YouTube가 값을 제공하지 않아 전부 비어 있거나 0입니다. 분석에서 제외합니다.

## 2. 채널 합계와 영상별 합계의 불일치 → 삭제 영상(D)

현재 확인 가능한 9개 영상의 합계는 채널 합계와 맞지 않습니다. YouTube는 삭제된 영상의 기록도 채널 합계에 남기므로, 차이는 저작권 문제로 **삭제된 영상들(D)**의 몫입니다. 삭제가 확인된 영상은 1개(예능 하이라이트 편집, 4분 32초)이며, 그 밖에 삭제된 영상이 있었는지는 확인되지 않습니다. 아래 D 값은 현존하지 않는 영상 전체의 합계입니다.

In [3]:
ch = dict(views=int(total['유효 조회수']), watch_h=float(total['시청 시간(단위: 시간)']),
          impr=int(total['썸네일 노출수']), subs=int(total['구독자']))
d1 = {k: float(ch[k] - v[k].sum()) for k in ch}
d1['avg_view_s'] = d1['watch_h'] * 3600 / d1['views']
d1['ctr_upper'] = d1['views'] / d1['impr'] * 100   # 모든 조회가 노출 클릭에서 왔다고 가정한 상한
share = {k: d1[k] / ch[k] * 100 for k in ['views', 'watch_h', 'impr', 'subs']}
print('채널 합계:', ch)
print('삭제 영상(D) 합계:', {k: round(x, 1) for k, x in d1.items()})
print('삭제 영상(D) 비중(%):', {k: round(float(x), 1) for k, x in share.items()})

채널 합계: {'views': 123577, 'watch_h': 5749.6093, 'impr': 2868407, 'subs': 218}
삭제 영상(D) 합계: {'views': 72409.0, 'watch_h': 3275.6, 'impr': 2576003.0, 'subs': 141.0, 'avg_view_s': 162.9, 'ctr_upper': 2.8}
삭제 영상(D) 비중(%): {'views': 58.6, 'watch_h': 57.0, 'impr': 89.8, 'subs': 64.7}


## 3. 급증 구간 확인: 트래픽 소스 일별 데이터

2019년 3월 9–22일 일별 조회수가 평소(하루 수십 회)의 100배 이상으로 뛰었다가 3월 23일 급단절됩니다. 같은 기간을 유입 경로별로 나눕니다.

In [4]:
t = pd.read_csv(f'{RAW}/traffic_daily_2019_2021.csv', parse_dates=['날짜'])
t.columns = ['date', 'source', 'views']
t = t[(t.date >= START) & (t.date <= END)]
GROUP = {'추천 동영상': '추천·탐색', '탐색 기능': '추천·탐색', 'YouTube 검색': '검색'}
t['group'] = t.source.map(GROUP).fillna('기타')
in_spike = t.date.between(*SPIKE)
split = (t.assign(period=in_spike.map({True: '급증 2주', False: '그 외 기간'}))
          .pivot_table(index='period', columns='group', values='views', aggfunc='sum'))
split['합계'] = split.sum(axis=1)
pct = split[['추천·탐색', '검색', '기타']].div(split['합계'], axis=0).mul(100).round(1)
print(split); print(pct)
rec = t[t.source == '추천 동영상']
print('추천 동영상 유입 중 급증 2주 비중: %.1f%%' % (rec[rec.date.between(*SPIKE)].views.sum() / rec.views.sum() * 100))
spike_views = split.loc['급증 2주', '합계']
print('삭제 영상(D) 조회수 중 급증 2주 조회수의 비중: %.1f%%' % (spike_views / d1['views'] * 100))

group      검색    기타  추천·탐색     합계
period                           
그 외 기간  44155  5347   3439  52941
급증 2주    1170  1007  68459  70636
group   추천·탐색    검색    기타
period                   
그 외 기간    6.5  83.4  10.1
급증 2주    96.9   1.7   1.4
추천 동영상 유입 중 급증 2주 비중: 98.7%
삭제 영상(D) 조회수 중 급증 2주 조회수의 비중: 97.6%


In [5]:
daily = pd.read_csv(f'{RAW}/channel_daily_views_lifetime.csv', parse_dates=['날짜'])
daily.columns = ['date', 'views']
print(daily[daily.date.between('2019-03-05', '2019-03-26')].set_index('date').T)
# 급증 당시 이미 업로드돼 있던 현존 영상과, 그 영상의 같은 기간 일별 조회수
print('2019-03-09 이전 업로드된 현존 영상:', v.loc[pd.to_datetime(v.published) < '2019-03-09', 'code'].tolist())
top5 = pd.read_csv(f'{RAW}/top5_video_daily_lifetime.csv', parse_dates=['날짜'])
l1 = top5[top5['길이'] == 2131].set_index('날짜')['유효 조회수']
print('L1 일별 조회수 — 급증 2주: %d–%d회, 직전 4주 평균 %.0f회' % (l1[SPIKE[0]:SPIKE[1]].min(), l1[SPIKE[0]:SPIKE[1]].max(), l1['2019-02-09':'2019-03-08'].mean()))

date   2019-03-05  2019-03-06  2019-03-07  2019-03-08  2019-03-09  2019-03-10  2019-03-11  2019-03-12  2019-03-13  2019-03-14  ...  \
views          66          59          90         134        1091         471         386        1126        2649        3506  ...   

date   2019-03-17  2019-03-18  2019-03-19  2019-03-20  2019-03-21  2019-03-22  2019-03-23  2019-03-24  2019-03-25  2019-03-26  
views        6195        6224        6482        6195        7293       14278          46          48          24          27  

[1 rows x 22 columns]
2019-03-09 이전 업로드된 현존 영상: ['L1']
L1 일별 조회수 — 급증 2주: 18–68회, 직전 4주 평균 48회


**[사실]** 급증 2주 조회수의 대부분은 추천·탐색에서, 나머지 기간 조회수의 대부분은 검색에서 발생했다.
**[사실]** 급증은 삭제된 영상에서 발생했다. 근거: ① 급증 당시 이미 업로드돼 있던 현존 영상은 L1 하나뿐이며, L1의 같은 기간 일별 조회수는 평소 수준이었다. ② 3월 23일 조회수가 하루 만에 1/300로 급단절됐다. ③ 급증 2주 조회수가 삭제 영상(D) 전체 조회수의 97.6%를 차지한다. 따라서 D의 합계 지표는 사실상 급증 영상의 지표다.
**[추정]** 급증 영상은 삭제가 확인된 '예능 하이라이트 편집(4분 32초)'일 가능성이 높다. D의 평균 시청 시간(2분 43초)과 추천 유입 영상의 역산 길이(약 4.8분)가 이 영상의 길이와 맞는다. 다른 삭제 영상이 있었는지 확인할 수 없어 단정하지는 않는다.

## 4. 유입 경로별 퍼널 지표와 교란 요인(영상 길이)

In [6]:
ts = pd.read_csv(f'{RAW}/traffic_2019_2021.csv')
ts = ts.rename(columns={'트래픽 소스': 'source', '유효 조회수': 'views', '평균 조회율 (%)': 'pct_viewed',
                        '시청 시간(단위: 시간)': 'watch_h', '썸네일 노출수': 'impr', '썸네일 클릭률 (%)': 'ctr'})
ts['avg_view_s'] = ts['평균 시청 지속 시간'].map(hms)
ts = ts[['source', 'impr', 'ctr', 'views', 'avg_view_s', 'pct_viewed', 'watch_h']]
tot_impr = ts.loc[ts.source == '합계', 'impr'].iloc[0]; tot_views = ts.loc[ts.source == '합계', 'views'].iloc[0]
ts['impr_share'] = ts.impr / tot_impr * 100
ts['view_share'] = ts.views / tot_views * 100
# 평균 시청 시간 ÷ 평균 조회율 = 해당 경로로 시청된 영상의 평균 길이(역산)
ts['implied_len_min'] = ts.avg_view_s / (ts.pct_viewed / 100) / 60
main = ts[ts.source.isin(['추천 동영상', 'YouTube 검색', '탐색 기능'])].round(2)
main

,source,impr,ctr,views,avg_view_s,pct_viewed,watch_h,impr_share,view_share,implied_len_min
1,추천 동영상,2417384.0,1.84,57553,165,57.62,2638.12,84.28,46.57,4.77
2,YouTube 검색,263110.0,15.39,45325,143,10.44,1802.90,9.17,36.68,22.83
3,탐색 기능,144281.0,6.86,14345,185,37.16,740.79,5.03,11.61,8.30


**[사실]** 추천은 노출 비중이 크고 CTR이 낮으며 조회율이 높다. 검색은 노출 비중이 작고 CTR이 높으며 조회율이 낮다.
**주의:** 조회율 차이를 경로의 특성으로 해석하면 안 된다. 역산한 평균 영상 길이가 추천 약 4–5분, 검색 약 23분으로, **경로별로 소비된 영상 유형 자체가 다르다.** 조회율은 영상이 길수록 기계적으로 낮아지므로, 경로 비교에는 CTR과 노출 규모만 쓰고 조회율은 영상 길이를 통제한 뒤에 본다.

## 5. 영상 유형별 비교: 어떤 KPI를 쓰느냐에 따라 평가가 바뀐다

비교 대상: 2019–2021년 기간에 공개 상태였던 영상. 노출수 5,000회 미만 영상(P1)은 CTR 신뢰도가 낮아 제외합니다.

In [7]:
cmp = v[v.impr >= 5000].copy()
cmp['subs_per_1k'] = cmp.subs / cmp.views * 1000
cmp['watch_share'] = cmp.watch_h / ch['watch_h'] * 100
cols = ['code', 'type', 'length_s', 'impr', 'ctr', 'views', 'avg_view_s', 'pct_viewed', 'watch_h', 'watch_share', 'subs', 'subs_per_1k']
cmp = cmp[cols].sort_values('length_s', ascending=False).round(2)
cmp

,code,type,length_s,impr,ctr,views,avg_view_s,pct_viewed,watch_h,watch_share,subs,subs_per_1k
5,L3,롱폼 음성 콘텐츠,2260.0,29628,11.90,4114,273,12.10,312.46,5.43,3,0.73
2,L1,롱폼 음성 콘텐츠,2131.0,127652,17.04,25600,226,10.63,1610.79,28.02,31,1.21
4,L2,롱폼 음성 콘텐츠,1353.0,44764,8.72,4836,210,15.53,282.24,4.91,13,2.69
9,S1,콘셉트 숏폼(영상통화 형식),231.0,65837,19.12,14863,59,25.71,245.21,4.26,29,1.95
3,C1,예능 클립,64.0,12869,6.39,1005,50,79.11,14.14,0.25,1,1.00
6,C2,예능 클립,51.0,5352,3.59,249,32,63.21,2.23,0.04,1,4.02


In [8]:
long = cmp[cmp.code.str.startswith('L')]; short = cmp[cmp.length_s < 300]
print('롱폼 3개 시청 시간: %.0f시간 (채널의 %.1f%%)' % (long.watch_h.sum(), long.watch_h.sum() / ch['watch_h'] * 100))
print('롱폼 조회율 %.1f–%.1f%% / 평균 시청 %d–%d초' % (long.pct_viewed.min(), long.pct_viewed.max(), long.avg_view_s.min(), long.avg_view_s.max()))
print('5분 미만 영상 조회율 %.1f–%.1f%% / 평균 시청 %d–%d초' % (short.pct_viewed.min(), short.pct_viewed.max(), short.avg_view_s.min(), short.avg_view_s.max()))

롱폼 3개 시청 시간: 2205시간 (채널의 38.4%)
롱폼 조회율 10.6–15.5% / 평균 시청 210–273초
5분 미만 영상 조회율 25.7–79.1% / 평균 시청 32–59초


**[사실]** 조회율 기준으로는 짧은 영상이, 평균 시청 시간·총 시청 시간 기준으로는 롱폼이 우수하다.
**[가설]** 높은 CTR(L1 17%, S1 19%)은 검색 유입 비중이 높았기 때문일 수 있다. 영상별 트래픽 소스를 확보하지 않아 검증하지 못했다.

## 6. 전환(구독)

구독자 수는 **순증가(신규 − 취소)** 기준이며 값이 작아(0–31명) 해석에 주의가 필요합니다.

In [9]:
s = pd.read_csv(f'{RAW}/subscribers_daily_lifetime.csv', parse_dates=['날짜']); s.columns = ['date', 'subs']
s = s[s.date.between(START, END)]
sp = s[s.date.between(*SPIKE)].subs.sum(); other = s.subs.sum() - sp
vs = daily[daily.date.between(*SPIKE)].views.sum(); vo = daily[daily.date.between(START, END)].views.sum() - vs
print('급증 2주: 구독 +%d / 조회 %d → 1,000회당 %.2f명' % (sp, vs, sp / vs * 1000))
print('그 외 기간: 구독 %+d / 조회 %d → 1,000회당 %.2f명' % (other, vo, other / vo * 1000))
print('삭제 영상(D) 구독 기여: %+d명 (채널의 %.0f%%), 1,000회당 %.2f명' % (d1['subs'], share['subs'], d1['subs'] / d1['views'] * 1000))
vv = cmp[cmp.views >= 1000][['code', 'views', 'subs', 'subs_per_1k']]
print(); print('영상별 순증가 구독 (조회 1,000회 이상 영상):'); print(vv.to_string(index=False))
print('현존 영상 합계: 구독 %+d / 조회 %d → 1,000회당 %.2f명' % (v.subs.sum(), v.views.sum(), v.subs.sum() / v.views.sum() * 1000))

급증 2주: 구독 +205 / 조회 70636 → 1,000회당 2.90명
그 외 기간: 구독 +13 / 조회 52941 → 1,000회당 0.25명
삭제 영상(D) 구독 기여: +141명 (채널의 65%), 1,000회당 1.95명

영상별 순증가 구독 (조회 1,000회 이상 영상):
code  views  subs  subs_per_1k
  L3   4114     3         0.73
  L1  25600    31         1.21
  L2   4836    13         2.69
  S1  14863    29         1.95
  C1   1005     1         1.00
현존 영상 합계: 구독 +77 / 조회 51168 → 1,000회당 1.50명


**[사실]** 채널 일별 기준으로 보면 급증 2주의 1,000회당 순증가 구독자(2.90명)가 그 외 기간(0.25명)의 약 12배다.
**[사실]** 그러나 영상별 기준으로 보면 삭제 영상 D(약 1.95명)와 현존 주요 영상(1.2–2.7명)은 비슷한 범위다. 현존 영상만 합쳐도 +77명인데, 같은 기간 채널 일별 합계는 +13명에 그친다.
**[해석]** 채널 일별 순증가에는 기간 전체의 구독 취소가 섞이므로, 급증이 없는 시기의 전환이 과소평가된다. **같은 지표라도 집계 단위(채널 일별 vs 영상별)에 따라 결론이 달라진다.** 전환 규모를 키운 것은 전환율이 아니라 도달 규모였다고 보는 편이 데이터에 맞다.

## 7. 데이터 한계
- 비공개 이력이 있는 영상 6개 중 3개는 일별 데이터로 비공개 시점을 확인했으며(모두 2023년 이후), 분석 기간(2019–2021)에는 공개 상태였다. 나머지 3개는 시점을 확인하지 못했다.
- D의 지표는 채널 합계에서 현존 영상을 뺀 값으로, 확인된 삭제 영상 외에 다른 삭제 영상이 있었다면 그 몫도 포함된다.
- 영상별 트래픽 소스를 확보하지 않아, 경로 분석은 채널 전체 단위로만 해석한다.
- 구독자는 순증가 기준이며, 조회 1,000회 미만 영상의 구독 전환율은 1–2명 차이로 크게 흔들려 해석에서 제외했다.
- 영상 수가 적어 통계적 검정은 하지 않았다. 모든 결론은 사례 관찰이다.
- 분석은 채널 운영이 끝난 뒤(2026년)에 사후적으로 수행했다.

In [10]:
# 익명화된 결과 저장 (제목·ID 없음)
v.to_csv(f'{OUT}/videos_2019_2021_anon.csv', index=False)
main.to_csv(f'{OUT}/traffic_sources_2019_2021.csv', index=False)
pct.to_csv(f'{OUT}/traffic_mix_spike_vs_rest.csv')
m = t.assign(month=t.date.dt.to_period('M').astype(str)).pivot_table(index='month', columns='group', values='views', aggfunc='sum').fillna(0).astype(int)
m.to_csv(f'{OUT}/monthly_views_by_source.csv')
pd.Series({**{f'd1_{k}': round(x, 2) for k, x in d1.items()}, **{f'd1_share_{k}': round(x, 1) for k, x in share.items()}}).to_csv(f'{OUT}/deleted_video_estimate.csv', header=['value'])
m.tail(3)

group,검색,기타,추천·탐색
month,,,
2021-10,534,55,16
2021-11,395,44,14
2021-12,465,52,12
